# Reshaping and combining arrays

## Learning objectives

By the end of this notebook you will be able to:

- change an array's shape with reshape, flatten and ravel
- transpose arrays and add dimensions
- join arrays with concatenate and stack
- split arrays into parts
- sort arrays and find unique values

## Introduction

Data rarely arrives in the shape an analysis needs. Plate readers give 96 numbers in a line that belong
in an 8 × 12 grid; replicate runs come as separate arrays that need joining. This notebook covers
changing shape and combining arrays.

In [1]:
import numpy as np

## reshape

`reshape(rows, cols)` rearranges the same values into a new shape. The total size must stay the same.
`-1` means "work this dimension out for me".

In [2]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(4, -1))
try:
    a.reshape(5, 3)
except ValueError as e:
    print("ValueError:", e)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
ValueError: cannot reshape array of size 12 into shape (5,3)


### A 96-well plate

A plate reader exports 96 absorbance values in row order (A1, A2, ... A12, B1, ...). Reshaping gives the
plate layout:

In [3]:
rng = np.random.default_rng(11)
readings = rng.uniform(0.05, 1.2, 96).round(3)
plate = readings.reshape(8, 12)
rows = "ABCDEFGH"
print(plate.shape)
print("well C5:", plate[rows.index("C"), 5 - 1])
print("row means:", plate.mean(axis=1).round(3))

(8, 12)
well C5: 1.047
row means: [0.528 0.686 0.556 0.529 0.502 0.526 0.58  0.6  ]


## flatten and ravel

Both turn any array into 1-D. `flatten` always returns a copy; `ravel` returns a view when it can.

In [4]:
print(plate[:2, :3].flatten())
print(plate.ravel().shape)

[0.198 0.624 0.742 0.812 0.367 0.209]
(96,)


## Transpose

`.T` swaps rows and columns.

In [5]:
m = np.array([[1, 2, 3], [4, 5, 6]])
print(m.T, m.T.shape)

[[1 4]
 [2 5]
 [3 6]] (3, 2)


## Adding a dimension

Useful for broadcasting: turning a row of values into a column.

In [6]:
v = np.array([10, 20, 30])
print(v.shape, v[:, np.newaxis].shape, v[np.newaxis, :].shape, np.expand_dims(v, 1).shape)

(3,) (3, 1) (1, 3) (3, 1)


## Joining arrays

- `np.concatenate` joins along an existing axis
- `np.vstack` stacks rows (one array on top of another)
- `np.hstack` places arrays side by side
- `np.column_stack` turns 1-D arrays into the columns of a table
- `np.stack` creates a new axis

In [7]:
run1 = np.array([[5.1, 5.3], [6.0, 6.2]])
run2 = np.array([[5.0, 5.4], [6.1, 5.9]])
print(np.concatenate([run1, run2]))            # axis=0: more rows
print(np.concatenate([run1, run2], axis=1))    # more columns
print(np.vstack([run1, run2]).shape, np.hstack([run1, run2]).shape)
print(np.stack([run1, run2]).shape)            # (2 runs, 2, 2)

[[5.1 5.3]
 [6.  6.2]
 [5.  5.4]
 [6.1 5.9]]
[[5.1 5.3 5.  5.4]
 [6.  6.2 6.1 5.9]]
(4, 2) (2, 4)
(2, 2, 2)


In [8]:
age = np.array([34, 51, 47])
glucose = np.array([5.4, 7.8, 6.1])
table = np.column_stack([age, glucose])
print(table)

[[34.   5.4]
 [51.   7.8]
 [47.   6.1]]


Shapes must match along the joined edge:

In [9]:
try:
    np.vstack([np.ones((2, 3)), np.ones((2, 4))])
except ValueError as e:
    print("ValueError:", e)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 3 and the array at index 1 has size 4


## Splitting arrays

In [10]:
data = np.arange(10)
print(np.split(data, 2))
print(np.split(data, [3, 7]))             # cut at positions 3 and 7
print(np.array_split(data, 3))            # unequal parts allowed

left, right = np.hsplit(plate, 2)         # two halves of the plate
print(left.shape, right.shape)

[array([0, 1, 2, 3, 4]), array([5, 6, 7, 8, 9])]
[array([0, 1, 2]), array([3, 4, 5, 6]), array([7, 8, 9])]
[array([0, 1, 2, 3]), array([4, 5, 6]), array([7, 8, 9])]
(8, 6) (8, 6)


A train/test split, the first step in model evaluation, is just shuffling and splitting:

In [11]:
rng = np.random.default_rng(0)
idx = rng.permutation(20)
train_idx, test_idx = idx[:15], idx[15:]
print(sorted(train_idx.tolist()), sorted(test_idx.tolist()))

[0, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 16, 18, 19] [1, 9, 14, 15, 17]


## Sorting and unique values

In [12]:
values = np.array([3, 1, 2, 3, 3, 1])
print(np.sort(values))
print(np.sort(values)[::-1])               # descending
print(np.unique(values))
u, counts = np.unique(values, return_counts=True)
print(dict(zip(u.tolist(), counts.tolist())))

[1 1 2 3 3 3]
[3 3 3 2 1 1]
[1 2 3]
{1: 2, 2: 1, 3: 3}


In [13]:
results = np.array(["Neg", "Pos", "Neg", "Inc", "Neg", "Pos"])
print(np.unique(results, return_counts=True))

(array(['Inc', 'Neg', 'Pos'], dtype='<U3'), array([1, 3, 2]))


## Putting it together: replicate analysis

Three runs of 8 samples, measured in triplicate within each run. Stack the runs into a
3 × 8 × 3 array, then average over replicates and runs.

In [14]:
rng = np.random.default_rng(21)
true = np.linspace(1, 8, 8)
runs = [true[:, None] + rng.normal(0, 0.25, (8, 3)) + shift for shift in (0.0, 0.2, -0.1)]
cube = np.stack(runs)                    # (run, sample, replicate)
print(cube.shape)

per_run = cube.mean(axis=2)              # (3, 8): replicate means
print(per_run.round(2))
print("run effect:", (per_run.mean(axis=1) - per_run.mean()).round(2))
print("within-run CV %:", (cube.std(axis=2, ddof=1) / cube.mean(axis=2) * 100).mean().round(1))

(3, 8, 3)
[[1.01 2.07 2.82 4.17 4.86 6.36 7.07 7.95]
 [1.27 2.31 3.07 4.31 5.2  6.25 7.16 8.05]
 [0.87 2.03 2.79 3.81 4.79 5.98 6.89 7.73]]
run effect: [ 0.01  0.17 -0.17]
within-run CV %: 7.2


Run 2 reads highest and run 3 lowest, matching the shifts built into the simulation (+0.2 and −0.1
relative to run 1); random noise moves the estimates a little. The coefficient of variation (CV) summarises replicate precision as a percentage of the
mean.

## Practise

Exercises for this folder are in `exercises/NumPy_Exercises.ipynb`.

## Further reading

- [NumPy user guide](https://numpy.org/doc/stable/user/index.html)
- [Array manipulation routines](https://numpy.org/doc/stable/reference/routines.array-manipulation.html)
- [Sorting, searching and counting](https://numpy.org/doc/stable/reference/routines.sort.html)